# 📚 RAG System — Comprehensive Guide

**Retrieval-Augmented Generation (RAG)** is a technique that enhances LLM outputs by first retrieving relevant documents from a knowledge base, then feeding them as context to the language model. This notebook provides a **complete, line-by-line walkthrough** of a modular RAG system built from scratch.

---

## 🧠 What You'll Learn

| Step | Component | What It Does |
|------|-----------|--------------|
| 1 | **Reader** | Fetches raw text from a source (e.g., Wikipedia) |
| 2 | **Chunker** | Splits long text into smaller, searchable pieces |
| 3 | **Embedder** | Converts text chunks into numerical vectors |
| 4 | **Retriever** | Finds the most relevant chunks for a query |
| 5 | **LLM** | Generates an answer using retrieved context |

---

## 🏗️ High-Level RAG Architecture

```
                    ┌─────────────────────────────────────────────────────┐
                    │              RAG SYSTEM OVERVIEW                    │
                    └─────────────────────────────────────────────────────┘

  ┌──────────────┐     ┌──────────────┐     ┌──────────────────┐
  │  Knowledge   │     │    Chunk     │     │    Embedding     │
  │    Source    │ ──▶ │   into       │ ──▶ │     Model        │
  │ (Wikipedia)  │     │  segments    │     │ (sentence-trfmr) │
  └──────────────┘     └──────────────┘     └────────┬─────────┘
                                                      │
                                                      ▼
  ┌──────────────┐     ┌──────────────┐     ┌──────────────────┐
  │   User       │     │   Retrieve   │     │   Vector         │
  │   Query      │ ──▶ │  Top-K Most  │ ◀── │   Database       │
  │              │     │  Relevant    │     │   (Embeddings)   │
  └──────────────┘     └──────┬───────┘     └──────────────────┘
                               │
                               ▼
  ┌──────────────┐     ┌──────────────┐
  │   Final      │     │   LLM        │
  │   Answer     │ ◀── │  + Context   │
  │              │     │  Generate    │
  └──────────────┘     └──────────────┘
```

---

## ✅ Pros & Cons of This RAG Implementation (TL;DR)

| Pros | Cons |
|------|------|
| ✅ Modular design — swap any component | ❌ Relies on external APIs (Wikipedia, HF) |
| ✅ Clean abstract base classes & Protocol | ❌ No caching — re-fetches & re-embeds every run |
| ✅ Educational — easy to understand & extend | ❌ Basic chunking (character-based, no semantic boundaries) |
| ✅ Uses cosine similarity via matrix ops | ❌ No document metadata tracking (source, page) |
| ✅ Works with any HF embedding model | ❌ Single-threaded, no async |
| ✅ Includes overlap for chunk boundary continuity | ❌ No evaluation/quality metrics built-in |

*(A deeper breakdown appears at the end of the notebook.)*

In [ ]:
# ⚙️ Install Dependencies (run this cell first)
!pip install requests numpy huggingface-hub transformers


---
# 1. 📦 Imports & Setup

**Retrieval-Augmented Generation (RAG)** is a technique that enhances LLM outputs by first retrieving relevant documents from a knowledge base, then feeding them as context to the language model. This notebook provides a **complete, line-by-line walkthrough** of a modular RAG system built from scratch.

---

## 🧠 What You'll Learn

| Step | Component | What It Does |
|------|-----------|--------------|
| 1 | **Reader** | Fetches raw text from a source (e.g., Wikipedia) |
| 2 | **Chunker** | Splits long text into smaller, searchable pieces |
| 3 | **Embedder** | Converts text chunks into numerical vectors |
| 4 | **Retriever** | Finds the most relevant chunks for a query |
| 5 | **LLM** | Generates an answer using retrieved context |

---

## 🏗️ High-Level RAG Architecture

```
                    ┌─────────────────────────────────────────────────────┐
                    │              RAG SYSTEM OVERVIEW                    │
                    └─────────────────────────────────────────────────────┘

  ┌──────────────┐     ┌──────────────┐     ┌──────────────────┐
  │  Knowledge   │     │    Chunk     │     │    Embedding     │
  │    Source    │ ──▶ │   into       │ ──▶ │     Model        │
  │ (Wikipedia)  │     │  segments    │     │ (sentence-trfmr) │
  └──────────────┘     └──────────────┘     └────────┬─────────┘
                                                      │
                                                      ▼
  ┌──────────────┐     ┌──────────────┐     ┌──────────────────┐
  │   User       │     │   Retrieve   │     │   Vector         │
  │   Query      │ ──▶ │  Top-K Most  │ ◀── │   Database       │
  │              │     │  Relevant    │     │   (Embeddings)   │
  └──────────────┘     └──────┬───────┘     └──────────────────┘
                               │
                               ▼
  ┌──────────────┐     ┌──────────────┐
  │   Final      │     │   LLM        │
  │   Answer     │ ◀── │  + Context   │
  │              │     │  Generate    │
  └──────────────┘     └──────────────┘
```

---

## ✅ Pros & Cons of This RAG Implementation (TL;DR)

| Pros | Cons |
|------|------|
| ✅ Modular design — swap any component | ❌ Relies on external APIs (Wikipedia, HF) |
| ✅ Clean abstract base classes & Protocol | ❌ No caching — re-fetches & re-embeds every run |
| ✅ Educational — easy to understand & extend | ❌ Basic chunking (character-based, no semantic boundaries) |
| ✅ Uses cosine similarity via matrix ops | ❌ No document metadata tracking (source, page) |
| ✅ Works with any HF embedding model | ❌ Single-threaded, no async |
| ✅ Includes overlap for chunk boundary continuity | ❌ No evaluation/quality metrics built-in |

*(A deeper breakdown appears at the end of the notebook.)*

---
# 1. 📦 Imports & Setup

We start with the core dependencies:
- **`requests`** — to call Wikipedia and Hugging Face APIs
- **`numpy`** — for vector/matrix operations (embeddings, similarity)
- **`abc`** — to define abstract base classes (Reader, Chunker)
- **`typing.Protocol`** — structural typing for the embedding model

```
           ┌──────────┐
           │ requests │  ◀── HTTP calls to APIs
           ├──────────┤
  Imports  │  numpy   │  ◀── Numerical/vector ops
           ├──────────┤
           │   abc    │  ◀── Abstract base classes
           ├──────────┤
           │ Protocol │  ◀── Structural subtyping
           └──────────┘
```

In [ ]:
import requests
import numpy as np
from abc import ABC, abstractmethod
from typing import Protocol

# Polite User-Agent for Wikipedia API
USER_AGENT = "RAGSystemDemo/1.0 (Educational Project; Contact: student@example.com)"

**Why a custom User-Agent?** Wikipedia's API requires a descriptive User-Agent header. Without one, your requests may be blocked. This is good API citizenship.

---
# 2. 📖 Step 1 — Reading Content (Wikipedia)

The **Reader** component fetches raw text from an external knowledge source. Here we implement a `WikipediaReader`.

## 🔄 API Flow Diagram

```
  ┌────────────┐         ┌──────────────────┐         ┌─────────────┐
  │  Caller    │  page   │  WikipediaReader │  HTTP   │  Wikipedia  │
  │  requests  │ ──────▶ │  .__call__()     │ ──────▶ │   MediaWiki │
  │  content   │         │                  │         │    API      │
  └────────────┘         └──────────────────┘         └──────┬──────┘
       ▲                                                     │
       │                ┌──────────────────┐                 │
       └────────────────│  get_content_    │◀────────────────┘
                        │  from_wikipage() │    JSON with
                        │                  │    page extract
                        └──────────────────┘
```

### Two-Step API Call:
1. **Get Page ID** — resolve the page title to a numeric ID (handles redirects)
2. **Get Extract** — fetch the plain-text content using the page ID, strip HTML headings

### Why strip headings (`\n==`)?
Wikipedia sections like `== History ==` are structural markers, not content. Removing them keeps chunks clean.

In [ ]:
def get_content_from_wikipage(page: str) -> str:
    """Fetch full page content from Wikipedia API with proper User-Agent."""
    # Step 1: Get the page ID
    params = {
        "action": "query",
        "titles": page,
        "format": "json",
        "redirects": 1,
    }
    headers = {"User-Agent": USER_AGENT}
    response = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=headers, timeout=15)
    response.raise_for_status()
    data = response.json()
    pages = data["query"]["pages"]
    page_id = list(pages.keys())[0]
    if page_id == "-1":
        raise ValueError(f"Wikipedia page '{page}' not found.")

    # Step 2: Get the page content (extracts)
    params = {
        "action": "query",
        "pageids": page_id,
        "format": "json",
        "prop": "extracts",
        "explaintext": 1,  # Return plain text, not HTML
    }
    response = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=headers, timeout=15)
    response.raise_for_status()
    data = response.json()
    page_content = data["query"]["pages"][page_id]["extract"]

    # Clean up: remove headings
    DELIMITER_PARAGRAPH = "\n\n"
    HEADING_INDICATOR = "\n=="
    paragraphs = page_content.split(DELIMITER_PARAGRAPH)
    paragraphs_filtered = [p for p in paragraphs if not p.startswith(HEADING_INDICATOR)]
    return DELIMITER_PARAGRAPH.join(paragraphs_filtered)

**Key design decisions:**
- `redirects=1` — follows page redirects automatically (e.g., "ML" → "Machine Learning")
- `explaintext=1` — returns plain text instead of HTML, which is cleaner for NLP
- `timeout=15` — prevents hanging on slow network
- Stripping heading lines ensures chunks contain only readable content

---
# 3. ✂️ Step 2 — Chunking

LLMs have limited context windows. We need to split large documents into **chunks** that can be:
1. Embedded into vectors
2. Retrieved efficiently
3. Fitted into the LLM's prompt context

## 📐 Chunking Strategy Visualization

```
  Paragraph 1: ┌──────────────────────────────────────────────┐
               │  ... text ...                                │
               └──────────────────────────────────────────────┘

  Paragraph 2: ┌──────────────────────────────────────────────┐
               │  ... text ...                                │
               └──────┬───────────────────────────────────────┘
                      │ split into chunks of `max_chunk_chars`
                      ▼
          ┌──────────────────────┬──────────────────────┐
          │ Chunk A              │ Chunk B              │
          │ [0 : 500 chars]      │ [450 : 950 chars]    │
          │                      │                      │
          └──────────────────────┴──────────────────────┘
               ▲                            ▲
               │                            │
          50-char overlap ──────────────────┘
          (preserves context across boundaries)
```

### Strategy: Character-level chunking within paragraphs
- Split by paragraphs first (preserves topic boundaries)
- Within each paragraph, slide a window with **overlap**
- Overlap prevents cutting sentences/ideas in half

In [ ]:
def chunk_chars_within_paragraphs(text, paragraphs_indicator="\n\n", max_chunk_chars=500, overlap=50):
    """Split text into chunks: first by paragraphs, then by character window."""
    chunks = []
    for paragraph in text.split(paragraphs_indicator):
        curr_char_index = 0
        while True:
            start, end = curr_char_index, curr_char_index + max_chunk_chars
            if start >= len(paragraph):
                break
            chunks.append(paragraph[start:end].strip())
            if end >= len(paragraph):
                break
            curr_char_index += (max_chunk_chars - overlap)
    return chunks

**Why this approach?**

| Strategy | Pros | Cons |
|----------|------|------|
| **Character-based (this code)** | Simple, deterministic, fast | Ignores sentence boundaries; may cut mid-word |
| Sentence-based | Preserves natural boundaries | More complex parsing; uneven chunk sizes |
| Semantic (LLM-based) | Ideal chunks | Expensive, slow |

The 50-char overlap means adjacent chunks share context, reducing the chance of losing information at boundaries.

---
# 4. 🧩 Abstract Base Classes

We define interfaces for pluggable components. This is the **Strategy Pattern** — we can swap implementations without changing the pipeline.

```
         ┌──────────┐         ┌──────────┐
         │  Reader  │         │  Chunker │
         │ (ABC)    │         │ (ABC)    │
         ├──────────┤         ├──────────┤
         │ __call__ │         │ __call__ │
         └────┬─────┘         └────┬─────┘
              │ implements         │ implements
              ▼                    ▼
    ┌──────────────────┐  ┌──────────────────────────┐
    │ WikipediaReader  │  │ CharacterChunkerWithin  │
    │ (reads a page)   │  │ Paragraph (char-based)  │
    └──────────────────┘  └──────────────────────────┘
```

In [ ]:
class Reader(ABC):
    """Abstract base class for reading content from a source."""
    @abstractmethod
    def __call__(self, *args, **kwargs) -> str:
        pass

class WikipediaReader(Reader):
    """Reads plain text content from a Wikipedia page."""
    def __call__(self, page: str) -> str:
        return get_content_from_wikipage(page)


class Chunker(ABC):
    """Abstract base class for splitting text into chunks."""
    @abstractmethod
    def __call__(self, text: str) -> list[str]:
        pass

class CharacterChunkerWithinParagraph(Chunker):
    """Chunks text by paragraphs, then by character window with overlap."""
    def __init__(self, paragraphs_indicator="\n\n", max_chunk_chars=500, overlap=50):
        self.indicator = paragraphs_indicator
        self.max = max_chunk_chars
        self.overlap = overlap
    def __call__(self, text: str) -> list[str]:
        return chunk_chars_within_paragraphs(text, self.indicator, self.max, self.overlap)

**Why `__call__` instead of a named method?**  
Using `__call__` makes the objects **callable** — they behave like functions, which keeps the pipeline clean (`reader(page)` vs `reader.read(page)`). This is a common Python pattern.

---
# 5. 🔢 Step 3 — Embedding & Retrieval

This is the **heart of the RAG system**. We convert text chunks into numerical vectors (embeddings) and use **cosine similarity** to find the most relevant chunks for a given query.

## 📐 How Cosine Similarity Works

```
                    Vector Space (simplified to 2D)

                      ▲
                      │
                      │
                  q • │
                 /    │              Chunks that are
                / θ   │              semantically similar
        ───────/──────│──────────▶   have a SMALL angle θ
              /       │              (cos θ ≈ 1)
             /  c1•   │
            /         │
           ▼          │
          c2•         │
                      │

  cos_sim(q, c) = (q · c) / (||q|| × ||c||)

  where: · = dot product, ||·|| = vector magnitude

  Since we normalize all vectors to unit length:
    cos_sim(q, c) = q · c   (just the dot product!)
```

### EmbeddingRetriever Flow

```
  ┌──────────┐     ┌────────────────┐     ┌───────────────┐
  │  Chunks  │     │ EmbeddingModel │     │  Normalized   │
  │ [c1..cN] │ ──▶ │ .encode()      │ ──▶ │  Embeddings   │
  └──────────┘     └────────────────┘     └───────┬───────┘
                                                    │
  ┌──────────┐     ┌────────────────┐               │
  │  Query   │     │ EmbeddingModel │               │
  │    q     │ ──▶ │ .encode(q)     │ ──┐           │
  └──────────┘     └────────────────┘   │           │
                                         ▼           ▼
                                   ┌──────────────────────┐
                                   │  q · embeddings^T    │
                                   │  (matrix multiply)   │
                                   └──────────┬───────────┘
                                              ▼
                                   ┌──────────────────────┐
                                   │ argsort, take top_k  │
                                   └──────────────────────┘
```

### The Embedding Model Protocol

Instead of an abstract base class, we use **`typing.Protocol`** — this enables **structural subtyping** (duck typing). Any object with an `.encode()` method that accepts strings and returns vectors is a valid embedding model. This is more flexible than inheritance.

In [ ]:
class EmbeddingModel(Protocol):
    """Protocol (structural type) for any embedding model.
    Any object with an `encode` method matching this signature works."""
    def encode(self, sentences: str | list[str], *args, **kwargs):
        ...


class EmbeddingRetriever:
    """Manages embeddings and performs semantic search via cosine similarity."""
    
    def __init__(self, model: EmbeddingModel):
        self.model = model
    
    def embed(self, chunks: list[str]):
        """Compute and store normalized embeddings for all chunks."""
        self.chunks = np.array(chunks)
        self.embeddings = self.model.encode(chunks)
        # Normalize to unit length for fast cosine similarity
        self.embeddings_normed = self.embeddings / np.linalg.norm(self.embeddings, axis=-1, keepdims=True)
    
    def retrieve(self, query: str, top_k: int = 5) -> list[str]:
        """Retrieve top-k most similar chunks for a query."""
        embedded_query = self.model.encode(query)
        q_normed = embedded_query / np.linalg.norm(embedded_query)
        # Cosine similarity = dot product of normalized vectors
        similarities = q_normed @ self.embeddings_normed.T
        # Get indices of top-k most similar chunks
        return self.chunks[np.argsort(similarities)[::-1][:top_k]]

**Key mathematical insight:** 
`q_normed @ self.embeddings_normed.T` is a **matrix-vector dot product** that computes cosine similarity between the query and ALL chunks in one operation. NumPy vectorization makes this extremely fast.

- `similarities` is a 1D array of length `N` (one similarity score per chunk)
- `argsort(similarities)[::-1]` sorts in descending order (most similar first)
- `[:top_k]` takes the top-K results

---
# 6. 🤖 Step 4 — LLM Q&A Integration

The final step: feed the retrieved context into an LLM to generate a grounded answer.

## 🧾 Prompt Construction

```
  ┌─────────────────────────────────────────────────┐
  │  System: You are a helpful Q&A bot.             │
  │                                                 │
  │  User:                                         │
  │    Question: What is artificial intelligence?  │
  │    Context:                                    │
  │      [0]: AI is the simulation of human...     │
  │      [1]: The field was founded in 1956...     │
  │      [2]: Modern AI uses deep learning...      │
  │                                                 │
  │  Assistant: Answer:                            │
  └─────────────────────────────────────────────────┘
```

In [ ]:
class QALLM:
    """Q&A LLM that uses retrieved context to answer questions."""
    
    def __init__(self, hf_url: str, hf_token: str):
        import transformers
        self.url = hf_url
        self.token = hf_token
        # Load tokenizer for chat template formatting
        self.tokenizer = transformers.AutoTokenizer.from_pretrained(
            "/".join(hf_url.split("/")[-2:])
        )
    
    def query(self, question: str, context: list[str], **params) -> str:
        """Build a prompt with context and query the LLM."""
        prompt = (
            f"Question: {question}\nContext:\n"
            + "\n".join([f"[{i}]: {c}" for i, c in enumerate(context)])
        )
        chat = [
            {"role": "system", "content": "You are a helpful question answering bot."},
            {"role": "user", "content": prompt},
        ]
        final_p = self.tokenizer.apply_chat_template(
            chat, tokenize=False, add_generation_prompt=True
        ) + "Answer:"
        return query_hf_inference(final_p, self.url, self.token, **params)


def query_hf_inference(message, api_url, api_token, **params):
    """Send a prompt to Hugging Face Inference API and get generated text."""
    headers = {"Authorization": f"Bearer {api_token}"}
    response = requests.post(
        api_url, headers=headers,
        json={"inputs": message, "parameters": params}
    )
    return response.json()[0]["generated_text"]

**Why use the tokenizer for prompt formatting?**  
Different LLMs expect different chat templates (e.g., `<|user|>`, `[INST]`, `<start_of_turn>`). Using `apply_chat_template` ensures the prompt is correctly formatted for the specific model.

**Why add "Answer:" at the end?**  
This serves as a **generation prefix** — it cues the model to start producing the answer directly, rather than repeating the prompt.

---
# 7. 🔗 The Full RAG Pipeline

Finally, we wire everything together into a single `RAGSystem` class.

## 🏗️ Complete Pipeline Flow

```
                    ┌─────────────────────┐
                    │     RAGSystem       │
                    │                     │
                    │  ┌───────────────┐  │
   setup(context) ──┼─▶│  chunker()    │  │
                    │  └───────┬───────┘  │
                    │          ▼          │
                    │  ┌───────────────┐  │
                    │  │ retriever.    │  │
                    │  │ embed()       │  │
                    │  └───────────────┘  │
                    └─────────────────────┘

   query(question)  ┌─────────────────────┐
                  ──┼─▶│ retriever.       │
                    │  │ retrieve(question)│
                    │  └───────┬──────────┘
                    │          ▼            
                    │  ┌───────────────┐  │
                    │  │ qa_llm.query │  │
                    │  │ (question,    │  │
                    │  │  context)     │  │
                    │  └───────┬───────┘  │
                    │          ▼          │
                    │      Answer!        │
                    └─────────────────────┘
```

In [ ]:
class RAGSystem:
    """Orchestrates the full Retrieval-Augmented Generation pipeline."""
    
    def __init__(self, chunker, retriever, qa_llm):
        self.chunker = chunker
        self.retriever = retriever
        self.qa_llm = qa_llm
    
    def setup(self, context: str):
        """Index the knowledge base: chunk + embed."""
        self.retriever.embed(self.chunker(context))
    
    def query(self, question: str, top_k: int = 5, **llm_params):
        """Answer a question using the indexed knowledge base."""
        context = self.retriever.retrieve(question, top_k)
        return self.qa_llm.query(question, context, **llm_params)

**Design highlights:**
- `setup()` is separate from `query()` — index once, query many times
- Top-K is configurable per query (default 5)
- LLM parameters (temperature, max_tokens, etc.) are passed through via `**llm_params`
- The system is **fully modular** — any component can be swapped independently

---
# 8. 🎯 Demo — Putting It All Together

Let's see the system in action! We'll:
1. Set up Hugging Face API access
2. Read a Wikipedia page
3. Chunk, embed, and index the content
4. Retrieve relevant chunks for sample questions
5. Generate answers using an LLM

In [ ]:
# ─── CONFIG ────────────────────────────────────────────────────────────────
# Set your Hugging Face API token here
# Get one from: https://huggingface.co/settings/tokens
HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # ← Replace with your token

# Embedding model (for semantic search)
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# LLM model for Q&A (must be available on your HF Inference providers)
HF_MODEL = "google/gemma-2-2b-it"

# Wikipedia page to use as knowledge source
WIKIPEDIA_TOPIC = "Artificial Intelligence"

# Queries to ask
QUESTIONS = [
    "What is artificial intelligence?",
    "What are the main applications of AI?",
    "What are the risks and benefits of AI?",
]

In [ ]:
from huggingface_hub import InferenceClient


class HFEmbeddingModel:
    """Embedding model using HuggingFace InferenceClient."""
    def __init__(self, model_name: str, api_token: str | None = None):
        self.client = InferenceClient(model=model_name, token=api_token or HF_TOKEN)
    
    def encode(self, sentences: str | list[str], *args, **kwargs) -> np.ndarray:
        if isinstance(sentences, str):
            sentences = [sentences]
        result = self.client.feature_extraction(sentences)
        arr = np.array(result)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        return arr

In [ ]:
# Step 1: Read Wikipedia content
print("=" * 65)
print(f"  RAG System Demo — Source: Wikipedia / {WIKIPEDIA_TOPIC}")
print("=" * 65)

reader = WikipediaReader()
content = reader(WIKIPEDIA_TOPIC)
print(f"\n[1/4] ✅ Read {len(content):,} characters from Wikipedia")

# Step 2: Chunk the text
chunker = CharacterChunkerWithinParagraph(max_chunk_chars=500, overlap=50)
chunks = chunker(content)
print(f"[2/4] ✅ Created {len(chunks)} chunks")

print("\n  Sample chunks (first 3):")
for i, chunk in enumerate(chunks[:3]):
    print(f"    [{i}] {chunk[:120]}...")

In [ ]:
if HF_TOKEN and HF_TOKEN != "hf_YOUR_TOKEN_HERE":
    # Step 3: Embed
    print("\n[3/4] Generating embeddings via Hugging Face API...")
    embed_model = HFEmbeddingModel(EMBEDDING_MODEL, HF_TOKEN)
    retriever = EmbeddingRetriever(embed_model)
    retriever.embed(chunks)
    print(f"       ✅ Indexed {len(retriever.chunks)} chunks with embeddings")

    # Step 4: Retrieve
    print("\n" + "=" * 65)
    print("  🔍 Semantic Search Results (Retrieval Only)")
    print("=" * 65)
    
    for i, question in enumerate(QUESTIONS, 1):
        print(f"\n  --- Q{i}: {question} ---")
        top_chunks = retriever.retrieve(question, top_k=2)
        for j, chunk in enumerate(top_chunks, 1):
            print(f"    [{j}] {chunk[:200]}...")

    # Step 5: LLM Q&A
    print("\n" + "=" * 65)
    print("  🤖 LLM Q&A (with retrieved context)")
    print("=" * 65)
    
    try:
        client = InferenceClient(model=HF_MODEL, token=HF_TOKEN)
        for i, question in enumerate(QUESTIONS, 1):
            print(f"\n  --- Q{i}: {question} ---")
            context = retriever.retrieve(question, top_k=3)
            prompt = (
                f"Question: {question}\nContext:\n"
                + "\n".join(f"[{j}]: {c}" for j, c in enumerate(context))
            )
            messages = [
                {"role": "system", "content": "You are a helpful Q&A bot. Answer concisely using the context."},
                {"role": "user", "content": prompt},
            ]
            result = client.chat_completion(messages=messages, max_tokens=200, temperature=0.3)
            print(f"  Answer: {result.choices[0].message.content}")
    except Exception as e:
        print(f"\n  ⚠️  Could not reach the LLM ({HF_MODEL}).")
        print(f"  Reason: {str(e)[:200]}")
        print("\n  The RAG pipeline through retrieval is fully working!")
        print("  To enable the LLM Q&A step, configure your Hugging Face")
        print("  Inference providers at: https://huggingface.co/settings/inference")
else:
    print("\n⚠️  HF_TOKEN not configured.")
    print("   Set your token in the config cell above to run the full pipeline.")
    print("   Get a token at: https://huggingface.co/settings/tokens")

print("\n" + "=" * 65)
print("  ✅ Demo complete!")
print("=" * 65)

---
# 9. 📊 Deep Dive: Pros & Cons Analysis

## ✅ Strengths

| Aspect | Details |
|--------|---------|
| **Modular Architecture** | Each component (Reader, Chunker, Retriever, LLM) is independently swappable. Want to use a PDF reader instead of Wikipedia? Just create a new `Reader` subclass. |
| **Clean Abstractions** | `ABC` + `Protocol` gives both interface enforcement and flexible duck-typing. The `EmbeddingModel` Protocol means you can use any library (sentence-transformers, OpenAI, Cohere) without inheritance. |
| **Educational Value** | The code is simple enough to understand end-to-end — no framework magic, just clear Python + NumPy. |
| **Efficient Retrieval** | Uses vectorized NumPy operations (single matrix multiply for all similarity computations). No loop over chunks. |
| **Overlap in Chunking** | The 50-character overlap between chunks helps preserve context across boundaries. |
| **Proper Chat Templating** | Uses `apply_chat_template` for correct prompt formatting per model. |

## ❌ Limitations & Trade-offs

| Aspect | Details | Mitigation Ideas |
|--------|---------|------------------|
| **API Dependency** | Wikipedia and Hugging Face APIs must be reachable. No offline mode. | Add local file reader; cache API responses to disk. |
| **No Caching** | Every run re-fetches and re-embeds everything. | Add `@functools.lru_cache` or disk cache for embeddings. |
| **Basic Chunking** | Character-level chunking ignores sentence/word boundaries. A chunk might contain half a sentence. | Use `spaCy` or `nltk` for sentence-aware chunking. |
| **No Metadata** | Chunks are plain strings — no tracking of which page, paragraph, or section they came from. | Store chunks as `dict` with `{"text": ..., "source": ..., "index": ...}`. |
| **No Persistence** | Embeddings live in memory only. Restart = re-embed. | Use FAISS or ChromaDB for persistent vector storage. |
| **No Async** | Single-threaded HTTP calls block execution. | Use `httpx.AsyncClient` for concurrent API calls. |
| **No Evaluation** | No metrics to measure retrieval or generation quality. | Add `Recall@K`, `MRR`, or LLM-as-judge evaluation. |
| **LLM Cost** | Each query calls an external API. For many queries, this adds up. | Use a local model via `llama.cpp` or `vLLM`. |

## 🎯 When to Use This Code

```
  ┌─────────────────────────────────────────────────────────────┐
  │  ✅ Use this when:                                         │
  │  • You're learning RAG concepts from scratch              │
  │  • You need a minimal, understandable codebase            │
  │  • You want to prototype with Wikipedia + Hugging Face    │
  │  • You're teaching others about RAG                       │
  │                                                           │
  │  ❌ Not ideal when:                                       │
  │  • You need production-scale performance (use LangChain)   │
  │  • You need persistent vector storage (use ChromaDB/FAISS) │
  │  • You have millions of documents (use distributed search) │
  │  • You need real-time streaming responses                  │
  └─────────────────────────────────────────────────────────────┘
```

---

# 🎓 Summary

You've now seen a **complete, modular RAG system** built from scratch:

| Component | What It Does | Key Technique |
|-----------|-------------|---------------|
| **WikipediaReader** | Fetches knowledge | MediaWiki API + text cleaning |
| **CharacterChunker** | Splits into pieces | Paragraph boundaries + sliding window with overlap |
| **EmbeddingRetriever** | Finds relevant chunks | Sentence embeddings + cosine similarity (normalized dot product) |
| **QALLM** | Generates answers | Chat template + HF Inference API |
| **RAGSystem** | Orchestrates pipeline | setup() → index, query() → retrieve + generate |

The key insight: **retrieval grounds the LLM in actual facts**, reducing hallucinations and enabling question-answering over domains the model wasn't trained on.